<a href="https://colab.research.google.com/github/ShanshanHoo/agentic-research-paper-assistant/blob/main/preprocessing_ign.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain langchain-community langchain-chroma langchain-openai

In [ ]:
import json
import os
from typing import List, Dict, Any
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = "data"
INPUT_FILE = os.path.join(DATA_DIR, "ign_feeds.json")


def load_ign_articles() -> List[Dict[str, Any]]:
    """
    Load IGN feed items from the JSON file created by ingestion.
    """
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def build_article_text(article: Dict[str, Any]) -> str:
    """
    Build the text that will actually be embedded.
    We keep useful context like title, summary, author, and publication date.
    """
    title = article.get("title", "").strip()
    summary = article.get("summary", "").strip()
    author = article.get("author", "").strip()
    published = article.get("published", "").strip()
    link = article.get("link", "").strip()

    parts = []
    if title:
        parts.append(f"Title: {title}")
    if summary:
        parts.append(f"Summary: {summary}")
    if author:
        parts.append(f"Author: {author}")
    if published:
        parts.append(f"Published: {published}")
    if link:
        parts.append(f"Link: {link}")

    return "\n".join(parts).strip()


def chunk_ign_articles(chunk_size: int = 1000, overlap: int = 200) -> List[Dict[str, Any]]:
    """
    Split each IGN article record into chunks for embedding.
    """
    articles = load_ign_articles()
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )

    chunks: List[Dict[str, Any]] = []

    for article in articles:
        article_text = build_article_text(article)
        if not article_text:
            continue

        splits = splitter.split_text(article_text)

        for idx, chunk in enumerate(splits):
            chunks.append({
                "title": article.get("title", ""),
                "summary": article.get("summary", ""),
                "author": article.get("author", ""),
                "published": article.get("published", ""),
                "link": article.get("link", ""),
                "feed_url": article.get("feed_url", ""),
                "source_type": article.get("source_type", "ign_news"),
                "chunk_id": idx,
                "chunk": chunk
            })

    return chunks


if __name__ == "__main__":
    chunks = chunk_ign_articles()
    print(f"Created {len(chunks)} chunks from IGN feed items.")